# ML Analaysis (Train/Test Split)

In [ ]:
## Notebook Summary

This notebook performs fault classification using a train/test split.

- Loads feature data from `stage4_features.csv`.
- Removes metadata and selected feature columns.
- Separates the target variable, `class_name`, from the input features.
- Splits the data into training and testing sets.
- Trains and evaluates:
    - Logistic Regression with standardized features.
    - Random Forest Classifier.
    - A 1D Convolutional Neural Network using PyTorch.
- Reports accuracy, confusion matrices, classification reports, and prediction time.
- Uses a validation split and early stopping during CNN training.
- Saves and reloads the best CNN model as `best_model.pth`.

Overall Random forest and Logistic Regression predict the best in terms of acceracy with Random forest being slightly better. For throughput, linear regression was by far the fastest.

In [24]:
# random forest and logisitc regression libraries
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

# device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print device
print(f'Using device: {device}')

Using device: cuda


In [25]:
# Load the data
df = pd.read_csv("stage4_features.csv")

df.head()

,scenario_id,split,class_name,send_v_rms_a_v,send_v_rms_b_v,send_v_rms_c_v,send_i_rms_a_a,send_i_rms_b_a,send_i_rms_c_a,send_v_rms_ratio_a,...,receive_v_spectral_entropy_mean,receive_v_spectral_entropy_max,receive_v_fundamental_residual_ratio_mean,receive_v_crest_factor_mean,receive_v_flatline_fraction_mean,receive_i_spectral_entropy_mean,receive_i_spectral_entropy_max,receive_i_fundamental_residual_ratio_mean,receive_i_crest_factor_mean,receive_i_flatline_fraction_mean
0,stage4_006324,train,CA,116648.243328,132606.498764,117525.522755,2036.110915,226.360257,2063.679653,0.882780,...,0.115058,0.118373,0.055208,1.378418,0.010860,0.114468,0.116752,0.054456,1.608260,0.003342
1,stage4_000708,train,Healthy,132162.440207,132008.443957,131239.898341,174.127951,173.272842,173.452631,1.000069,...,0.113745,0.117383,0.013537,1.426633,0.001671,0.113762,0.116331,0.013120,1.424010,0.000835
2,stage4_003544,train,CG,131777.450133,132478.795791,77860.328031,235.813976,263.024186,2145.019828,0.999050,...,0.185925,0.330113,0.209468,1.618260,0.010025,0.220438,0.428927,0.057327,1.502808,0.001671
3,stage4_002495,validation,BG,131254.093122,80240.971916,130988.264764,327.626774,3477.002206,300.159245,0.998033,...,0.163333,0.260361,0.165770,1.637747,0.013367,0.164593,0.263828,0.044094,1.449196,0.006683
4,stage4_004132,train,AB,131671.968270,124426.084239,132221.749750,1327.319630,1300.746614,200.872834,0.990787,...,0.115309,0.117159,0.026249,1.438159,0.004177,0.114655,0.119182,0.029347,1.424801,0.002506


In [26]:
# Drop Information Columns
cols_to_drop = ['scenario_id', 'split']
df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

df.head()

,class_name,send_v_rms_a_v,send_v_rms_b_v,send_v_rms_c_v,send_i_rms_a_a,send_i_rms_b_a,send_i_rms_c_a,send_v_rms_ratio_a,send_v_rms_ratio_b,send_v_rms_ratio_c,...,receive_v_spectral_entropy_mean,receive_v_spectral_entropy_max,receive_v_fundamental_residual_ratio_mean,receive_v_crest_factor_mean,receive_v_flatline_fraction_mean,receive_i_spectral_entropy_mean,receive_i_spectral_entropy_max,receive_i_fundamental_residual_ratio_mean,receive_i_crest_factor_mean,receive_i_flatline_fraction_mean
0,CA,116648.243328,132606.498764,117525.522755,2036.110915,226.360257,2063.679653,0.882780,1.003292,0.889548,...,0.115058,0.118373,0.055208,1.378418,0.010860,0.114468,0.116752,0.054456,1.608260,0.003342
1,Healthy,132162.440207,132008.443957,131239.898341,174.127951,173.272842,173.452631,1.000069,0.999684,0.999028,...,0.113745,0.117383,0.013537,1.426633,0.001671,0.113762,0.116331,0.013120,1.424010,0.000835
2,CG,131777.450133,132478.795791,77860.328031,235.813976,263.024186,2145.019828,0.999050,0.999888,0.594339,...,0.185925,0.330113,0.209468,1.618260,0.010025,0.220438,0.428927,0.057327,1.502808,0.001671
3,BG,131254.093122,80240.971916,130988.264764,327.626774,3477.002206,300.159245,0.998033,0.614567,1.000208,...,0.163333,0.260361,0.165770,1.637747,0.013367,0.164593,0.263828,0.044094,1.449196,0.006683
4,AB,131671.968270,124426.084239,132221.749750,1327.319630,1300.746614,200.872834,0.990787,0.945380,0.999209,...,0.115309,0.117159,0.026249,1.438159,0.004177,0.114655,0.119182,0.029347,1.424801,0.002506


In [27]:
# Split the data into features and target variable

X = df.drop(columns=['class_name',
                     'send_v_rms_ratio_a', 'send_v_rms_ratio_b', 'send_v_rms_ratio_c', 
                     'send_i_rms_ratio_a', 'send_i_rms_ratio_b', 'send_i_rms_ratio_c', 
                     'send_power_factor', 'send_active_power_w', 'send_reactive_power_var',
                     'receive_v_rms_a_v', 'receive_v_rms_b_v', 'receive_v_rms_c_v', 
                     'receive_i_rms_a_a', 'receive_i_rms_b_a', 'receive_i_rms_c_a', 
                     'receive_v_rms_ratio_a', 'receive_v_rms_ratio_b', 'receive_v_rms_ratio_c', 
                     'receive_i_rms_ratio_a', 'receive_i_rms_ratio_b', 'receive_i_rms_ratio_c', 
                     'receive_v0_rms_v', 'receive_v1_rms_v', 'receive_v2_rms_v', 
                     'receive_i0_rms_a', 'receive_i1_rms_a', 'receive_i2_rms_a', 
                     'receive_v0_v1_ratio', 'receive_v2_v1_ratio', 'receive_i0_i1_ratio', 
                     'receive_i2_i1_ratio', 'receive_active_power_w', 'receive_reactive_power_var', 
                     'receive_power_factor', 'receive_z_magnitude_a_ohm', 'receive_z_magnitude_b_ohm', 
                     'receive_z_magnitude_c_ohm', 'receive_v_rms_imbalance_pct', 'receive_i_rms_imbalance_pct', 
                     'send_v_spectral_entropy_mean', 'send_v_spectral_entropy_max', 'send_v_fundamental_residual_ratio_mean', 
                     'send_v_crest_factor_mean', 'send_v_flatline_fraction_mean', 'send_i_spectral_entropy_mean', 
                     'send_i_spectral_entropy_max', 'send_i_fundamental_residual_ratio_mean', 'send_i_crest_factor_mean', 
                     'send_i_flatline_fraction_mean', 'receive_v_spectral_entropy_mean', 
                     'receive_v_spectral_entropy_max', 'receive_v_fundamental_residual_ratio_mean', 
                     'receive_v_crest_factor_mean', 'receive_v_flatline_fraction_mean', 'receive_i_spectral_entropy_mean', 
                     'receive_i_spectral_entropy_max', 'receive_i_fundamental_residual_ratio_mean', 'receive_i_crest_factor_mean', 
                     'receive_i_flatline_fraction_mean'], errors='ignore')

y = df['class_name']


# Number of classes
num_classes = y.nunique()

In [28]:
# train and test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Logisitic Regression Model

In [29]:
# Logistic Regression Model
lr_model = LogisticRegression(solver='lbfgs', max_iter=200)
# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the model to the training data
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Report Time
# Start the timer
start_time = time.perf_counter()
# Run model prediction
predictions = lr_model.predict(X_test_scaled)
# End the timer
end_time = time.perf_counter()

duration = end_time - start_time
print(f"Model ran in {duration:.4f} seconds")




Accuracy Score: 0.9945454545454545

Confusion Matrix:
 [[209   0   0   0   0   0   0   0   0   0   0]
 [  0 194   0   0   0   0   0   0   0   0   1]
 [  4   0 189   0   0   0   0   0   0   0   0]
 [  0   0   0 191   0   0   0   0   0   0   0]
 [  0   0   0   0 204   0   0   0   0   0   0]
 [  0   0   0   0   3 203   0   0   0   0   0]
 [  0   0   0   0   0   0 175   0   0   0   0]
 [  0   0   0   0   0   0   0 216   1   0   0]
 [  0   0   0   0   0   0   0   3 200   0   0]
 [  0   0   0   0   0   0   0   0   0 203   0]
 [  0   0   0   0   0   0   0   0   0   0 204]]

Classification Report:
               precision    recall  f1-score   support

          AB       0.98      1.00      0.99       209
         ABC       1.00      0.99      1.00       195
         ABG       1.00      0.98      0.99       193
          AG       1.00      1.00      1.00       191
          BC       0.99      1.00      0.99       204
         BCG       1.00      0.99      0.99       206
          BG       1.00

In [30]:
# random forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)


print("Accuracy Score:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

# Report Time
# Start the timer
start_time = time.perf_counter()
# Run model prediction
predictions = rf_model.predict(X_test)
# End the timer
end_time = time.perf_counter()

duration = end_time - start_time
print(f"Model ran in {duration:.4f} seconds")


Accuracy Score: 0.9972727272727273

Confusion Matrix:
 [[207   0   2   0   0   0   0   0   0   0   0]
 [  0 195   0   0   0   0   0   0   0   0   0]
 [  2   0 191   0   0   0   0   0   0   0   0]
 [  0   0   0 191   0   0   0   0   0   0   0]
 [  0   0   0   0 204   0   0   0   0   0   0]
 [  0   0   0   0   0 206   0   0   0   0   0]
 [  0   0   0   0   0   0 175   0   0   0   0]
 [  0   0   0   0   0   0   0 215   2   0   0]
 [  0   0   0   0   0   0   0   0 203   0   0]
 [  0   0   0   0   0   0   0   0   0 203   0]
 [  0   0   0   0   0   0   0   0   0   0 204]]

Classification Report:
               precision    recall  f1-score   support

          AB       0.99      0.99      0.99       209
         ABC       1.00      1.00      1.00       195
         ABG       0.99      0.99      0.99       193
          AG       1.00      1.00      1.00       191
          BC       1.00      1.00      1.00       204
         BCG       1.00      1.00      1.00       206
          BG       1.00

## 1D CNN Model

### Define 1d Convolution

In [31]:
class Conv1DFaultClassifier(nn.Module):
    def __init__(self, num_classes):
        super(Conv1DFaultClassifier, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = torch.max(x, dim=2)[0]
        x = self.fc(x)
        return x


model = Conv1DFaultClassifier(num_classes).to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters())}')

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

Conv1DFaultClassifier(
  (conv1): Sequential(
    (0): Conv1d(1, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=11, bias=True)
  )
)

Total parameters: 33931
X_train shape: (8800, 21)
y_train shape: (8800,)
X_test shape: (2200, 21)
y_test shape: (2200,)


### Train Model

In [ ]:
# Addded validation split since we are using early stopping in the training loop.
# Keep the test split untouched. Use part of the training split for validation.
X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

cnn_scaler = StandardScaler()
X_train_cnn_scaled = cnn_scaler.fit_transform(X_train_cnn).astype(np.float32)
X_val_cnn_scaled = cnn_scaler.transform(X_val_cnn).astype(np.float32)
X_test_cnn_scaled = cnn_scaler.transform(X_test).astype(np.float32)

label_encoder = LabelEncoder()
y_train_cnn_encoded = label_encoder.fit_transform(y_train_cnn)
y_val_cnn_encoded = label_encoder.transform(y_val_cnn)
y_test_cnn_encoded = label_encoder.transform(y_test)

X_train_tensor = torch.tensor(X_train_cnn_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_cnn_encoded, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_cnn_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_cnn_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_cnn_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_cnn_encoded, dtype=torch.long)

# Create DataLoaders for all splits
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=256)

# Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop Variables
epochs = 500
best_val_acc = 0
patience = 10
no_improve = 0

print(f"=== GPU Status ===")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Training & Validation Loop
for epoch in range(epochs):
    epoch_start = time.time()
    
    # Training Phase
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # Evaluation Phase
    model.eval()
    val_preds = []
    test_preds = []
    
    with torch.no_grad():
        # Batched Validation
        for X_batch, _ in val_loader:
            outputs = model(X_batch.to(device))
            val_preds.append(torch.argmax(outputs, dim=1).cpu())
        
        # Batched Testing
        for X_batch, _ in test_loader:
            outputs = model(X_batch.to(device))
            test_preds.append(torch.argmax(outputs, dim=1).cpu())

    # Combine batch predictions
    val_preds_all = torch.cat(val_preds).numpy()
    test_preds_all = torch.cat(test_preds).numpy()
    
    # Calculate Metrics
    val_acc = accuracy_score(y_val_cnn_encoded, val_preds_all)
    test_acc = accuracy_score(y_test_cnn_encoded, test_preds_all)
    
    epoch_time = time.time() - epoch_start
    print(f'Epoch {epoch+1}/{epochs}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}, Time: {epoch_time:.2f}s')

    # Early Stopping & Model Saving
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break


model.load_state_dict(torch.load('best_model.pth', weights_only=True))

=== GPU Status ===
Device: cuda | GPU: NVIDIA RTX A2000 Laptop GPU
Epoch 1/500, Val Acc: 0.6301, Test Acc: 0.6286, Time: 0.70s
Epoch 2/500, Val Acc: 0.8824, Test Acc: 0.8800, Time: 0.13s
Epoch 3/500, Val Acc: 0.9636, Test Acc: 0.9568, Time: 0.13s
Epoch 4/500, Val Acc: 0.9756, Test Acc: 0.9714, Time: 0.15s
Epoch 5/500, Val Acc: 0.9869, Test Acc: 0.9823, Time: 0.15s
Epoch 6/500, Val Acc: 0.9898, Test Acc: 0.9868, Time: 0.18s
Epoch 7/500, Val Acc: 0.9909, Test Acc: 0.9914, Time: 0.14s
Epoch 8/500, Val Acc: 0.9903, Test Acc: 0.9886, Time: 0.14s
Epoch 9/500, Val Acc: 0.9920, Test Acc: 0.9927, Time: 0.13s
Epoch 10/500, Val Acc: 0.9909, Test Acc: 0.9932, Time: 0.13s
Epoch 11/500, Val Acc: 0.9932, Test Acc: 0.9932, Time: 0.15s
Epoch 12/500, Val Acc: 0.9892, Test Acc: 0.9932, Time: 0.15s
Epoch 13/500, Val Acc: 0.9926, Test Acc: 0.9945, Time: 0.16s
Epoch 14/500, Val Acc: 0.9926, Test Acc: 0.9945, Time: 0.14s
Epoch 15/500, Val Acc: 0.9915, Test Acc: 0.9941, Time: 0.14s
Epoch 16/500, Val Acc: 0.99

<All keys matched successfully>